In [72]:
#import libraries
from pathlib import Path
import sqlite3
import json

import pandas as pd
import requests
from io import StringIO

#define global variables
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

In [3]:
#create source dataset
participants = pd.DataFrame({
    "id": ["001", "002", "003", "004", "005"],
    "age": [24, 31, 27, 42, 35],
    "group": ["A", "B", "A", "B", "A"],
    "score": [81, 94, 88, 91, 85]
})

participants

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


In [4]:
participants.dtypes

id       object
age       int64
group    object
score     int64
dtype: object

The ID and group are strings, while age and score are integers. Even if it's numeric, an ID cannot be averaged or have numerical summaries be performed on it, because it would be meaningless, so it should be a string that would be treated as an independent grouping instead.

In [14]:
csv_path = Path("../" + str(RAW_DIR) + "/participants.csv")

participants.to_csv(
    csv_path,
    index=False
)

csv_path.exists()

True

In the CSV, it saves the values only separated by commas. No types are saved, but each column does retain that type's qualities. 

In [16]:
#read in the df
loaded = pd.read_csv(csv_path)

loaded

,id,age,group,score
0,1,24,A,81
1,2,31,B,94
2,3,27,A,88
3,4,42,B,91
4,5,35,A,85


In [18]:
loaded.dtypes

id        int64
age       int64
group    object
score     int64
dtype: object

The ID is read in as an integer, so zero padding is removed for sake of simplicity. This is different from previously, where they were strings.

In [19]:
loaded = pd.read_csv(
    csv_path,
    dtype={
        "id": "string",
        "group": "category"
    }
)

loaded

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


In [20]:
loaded.dtypes

id       string[python]
age               int64
group          category
score             int64
dtype: object

ID is now a string again, with the zero padding, and the group variable is now a cateogry, which can be subset according to unique values.

In [23]:
messy = pd.DataFrame({
    "id": ["001", "002", "003", "004"],
    "score": ["81", "-999", "88", "NA"]
})

messy_path = Path( "../" + str(RAW_DIR) + "/messy_scores.csv")

messy.to_csv(
    messy_path,
    index=False
)

#read the data back in
messy_loaded = pd.read_csv(
    messy_path,
    na_values=["NA", "-999"]
)

In [24]:
messy_loaded

,id,score
0,1,81.0
1,2,NaN
2,3,88.0
3,4,NaN


In [25]:
messy_loaded.dtypes

id         int64
score    float64
dtype: object

In [26]:
messy_loaded.isna().sum()

id       0
score    2
dtype: int64

Loading dataframes can misassign missing values, so it's necessary to inspect both the data file and the resulting read-in dataframe to make sure all data is being handled properly.

In [27]:
loaded.shape
loaded.head()
loaded.dtypes
loaded.isna().sum()
loaded["id"].nunique()
loaded["group"].value_counts()

group
A    3
B    2
Name: count, dtype: int64

In [28]:
assert len(loaded) == 5
assert loaded["id"].is_unique

In [29]:
assert set(
    loaded["group"].astype(str)
) <= {"A", "B"}

In [39]:
# SQLite Database
db_path = Path( "../" + str(RAW_DIR) + "/study.db")

connection = sqlite3.connect(db_path)

In [40]:
participants.to_sql(
    "participants",
    connection,
    if_exists="replace",
    index=False
)

5

In [32]:
sql_df = pd.read_sql(
    "SELECT * FROM participants",
    connection
)

sql_df

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


Pandas can read in data, but the type is not stored, so it gives its best guess. This is different from an SQLite database, since the databse can store types and reading it in preserves that.

In [33]:
query = """
SELECT
    id,
    age,
    score
FROM participants
"""

selected = pd.read_sql(
    query,
    connection
)

In [34]:
participants[
    ["id", "age", "score"]
]

,id,age,score
0,001,24,81
1,002,31,94
2,003,27,88
3,004,42,91
4,005,35,85


In [36]:
query = """
SELECT *
FROM participants
WHERE age >= 30
"""

older = pd.read_sql(
    query,
    connection
)

participants.loc[
    participants["age"] >= 30
]

,id,age,group,score
1,002,31,B,94
3,004,42,B,91
4,005,35,A,85


The commands form the same outputs, but the Pandas commands are more straightforward. This of course depends on what data storgae is possible for a given scenario. SQL might be easier for a large databse, however, since it doesn't need to be stored locally compared to a dataframe.

In [41]:
query = """
SELECT
    group,
    COUNT(*) AS n,
    AVG(score) AS mean_score
FROM participants
GROUP BY group
"""

summary = pd.read_sql(
    query,
    connection
)

DatabaseError: Execution failed on sql '
SELECT
    group,
    COUNT(*) AS n,
    AVG(score) AS mean_score
FROM participants
GROUP BY group
': near "group": syntax error

In [ ]:
query = """
SELECT *
FROM participants
WHERE group = ?
"""

group_a = pd.read_sql(
    query,
    connection,
    params=("A",)
)

Parameters allow commands to be altered without needing to alter the command itself, which can be helpful for looping or large queries.

In [43]:
outcomes = pd.DataFrame({
    "id": ["001", "002", "003", "005"],
    "followup_score": [85, 97, 91, 89]
})

outcomes.to_sql(
    "outcomes",
    connection,
    if_exists="replace",
    index=False
)

4

In [45]:
query = """
SELECT
    p.id,
    p.age,
    p.group,
    p.score,
    o.followup_score
FROM participants AS p
LEFT JOIN outcomes AS o
    ON p.id = o.id
"""

joined = pd.read_sql(
    query,
    connection
)

DatabaseError: Execution failed on sql '
SELECT
    p.id,
    p.age,
    p.group,
    p.score,
    o.followup_score
FROM participants AS p
LEFT JOIN outcomes AS o
    ON p.id = o.id
': near "group": syntax error

In [ ]:
joined
joined.isna().sum()

This is equivelent to a pandas Left Join 

In [ ]:
# close SQL Connection
connection.close()

Closing a connection is important to release memory that could be used otherwise. It's also good practice to ensure any changes are saved.

In [46]:
# HTTP Request
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(
    url,
    timeout=10
)

In [47]:
response.status_code

200

In [48]:
response.raise_for_status()

You need a response to ensure the page exists before trying to diagnose any future problems or bugs.

In [49]:
response.text[:500]

'[\n  {\n    "id": 1,\n    "name": "Leanne Graham",\n    "username": "Bret",\n    "email": "Sincere@april.biz",\n    "address": {\n      "street": "Kulas Light",\n      "suite": "Apt. 556",\n      "city": "Gwenborough",\n      "zipcode": "92998-3874",\n      "geo": {\n        "lat": "-37.3159",\n        "lng": "81.1496"\n      }\n    },\n    "phone": "1-770-736-8031 x56442",\n    "website": "hildegard.org",\n    "company": {\n      "name": "Romaguera-Crona",\n      "catchPhrase": "Multi-layered client-server neural-'

In [50]:
data = response.json()

In [51]:
type(data)
len(data)
type(data[0])
data[0]

{'id': 1,
 'name': 'Leanne Graham',
 'username': 'Bret',
 'email': 'Sincere@april.biz',
 'address': {'street': 'Kulas Light',
  'suite': 'Apt. 556',
  'city': 'Gwenborough',
  'zipcode': '92998-3874',
  'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
 'phone': '1-770-736-8031 x56442',
 'website': 'hildegard.org',
 'company': {'name': 'Romaguera-Crona',
  'catchPhrase': 'Multi-layered client-server neural-net',
  'bs': 'harness real-time e-markets'}}

The JSON become a python dictionary.

In [52]:
users = pd.DataFrame(data)

users.head()
users.columns

Index(['id', 'name', 'username', 'email', 'address', 'phone', 'website',
       'company'],
      dtype='object')

In [53]:
users.loc[0, "address"]

{'street': 'Kulas Light',
 'suite': 'Apt. 556',
 'city': 'Gwenborough',
 'zipcode': '92998-3874',
 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}

This column contains several pieces of data, meaning it cannot be reasonable concatonated to a single value.

In [78]:
users_flat = pd.json_normalize(data)

users_flat.head()
users_flat.columns

Index(['id', 'name', 'username', 'email', 'phone', 'website', 'address.street',
       'address.suite', 'address.city', 'address.zipcode', 'address.geo.lat',
       'address.geo.lng', 'company.name', 'company.catchPhrase', 'company.bs'],
      dtype='object')

In [55]:
json_path = Path( "../" + str(RAW_DIR) + "/users.json")

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        data,
        f,
        indent=2
    )

In [56]:
with open(
    json_path,
    "r",
    encoding="utf-8"
) as f:
    local_data = json.load(f)

In [57]:
local_data == data

True

In [58]:
bad_url = (
    "https://jsonplaceholder.typicode.com/"
    "does-not-exist"
)

bad_response = requests.get(
    bad_url,
    timeout=10
)

In [59]:
bad_response.status_code

404

In [60]:
bad_response.raise_for_status()

HTTPError: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/does-not-exist

Failure handling is necessary to prepare for if a website changes URL or the location of data.

In [75]:
url = (
    "https://en.wikipedia.org/wiki/"
    "List_of_states_and_territories_"
    "of_the_United_States"
)

tables = pd.read_html(url)

HTTPError: HTTP Error 403: Forbidden

In [76]:
len(tables)
tables[0].head()

,0
0,This article is part of a series on
1,Political divisions of the United States
2,First level
3,State (Commonwealth) Federal district Territor...
4,Second level


In [ ]:
for i, table in enumerate(tables[:5]):
    print(
        i,
        table.shape,
        list(table.columns)[:4]
    )

tables[0] guesses that something in the table exists, when that may not be the case and break code that is meant to run no matter what.

In [80]:
processed_csv = Path(
    "../" + str(PROCESSED_DIR) + "/users_processed.csv"
)

users_flat.to_csv(
    processed_csv,
    index=False
)

In [83]:
parquet_path = Path(
    "../" + str(PROCESSED_DIR) + "/users_processed.parquet"
)

users_flat.to_parquet(
    parquet_path,
    index=False
)

ArrowKeyError: No type extension with name arrow.py_extension_type found

In [ ]:
users_parquet = pd.read_parquet(
    parquet_path
)

In [ ]:
users_parquet.head()
users_parquet.dtypes

The parquet engine is missing, asking for packages pyarrow and fastparquet.

In [82]:
processed_csv.stat().st_size
parquet_path.stat().st_size

FileNotFoundError: [WinError 2] The system cannot find the file specified: '..\\data\\processed\\users_processed.parquet'

In [ ]:
ratio = (
    parquet_path.stat().st_size
    / processed_csv.stat().st_size
)

ratio

No, small data types generall don't show the size differences data types can make.

In [84]:
csv_again = pd.read_csv(
    processed_csv
)

In [86]:
csv_again.dtypes

id                       int64
name                    object
username                object
email                   object
phone                   object
website                 object
address.street          object
address.suite           object
address.city            object
address.zipcode         object
address.geo.lat        float64
address.geo.lng        float64
company.name            object
company.catchPhrase     object
company.bs              object
dtype: object

In [87]:
metadata = {
    "source": "JSONPlaceholder users endpoint",
    "url": "https://jsonplaceholder.typicode.com/users",
    "retrieved_for": "STA 556 Week 5",
    "raw_file": str(json_path),
    "processed_file": str(parquet_path)
}

In [90]:
metadata_path = Path(
    "../" + str(RAW_DIR) + "/users_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

You could include data collection/extraction date, the number of observations, and other information that could be helpful prior to reading in the data.